In [6]:
from dotenv import load_dotenv
from os import getenv
import vk_utils
from postgres_utils import connect_pgsql


import importlib
importlib.reload(vk_utils)


load_dotenv()

True

In [ ]:
new_communities = vk_utils.get_new_vk_communities(api_key=getenv(
    "VK_API_KEY"), queries=["подслушано", "район"], state_dsn=getenv("POSTGRESQL_DSN"))
print("Найдено сообществ:", len(new_communities))
new_communities[:10]

Найдено сообществ: 195


[{'id': 140549268, 'name': 'Подслушано Каршеринг', 'category': 'Автомобили'},
 {'id': 143277166,
  'name': 'ПОДСЛУШАНО НА СВО',
  'category': 'Дискуссионный клуб'},
 {'id': 224949751,
  'name': 'ПОДСЛУШАНО НА СВО / Выплаты, Поиск пропавших',
  'category': 'Дискуссионный клуб'},
 {'id': 185347562,
  'name': 'Подслушано ФСИН России',
  'category': 'Группа коллег'},
 {'id': 99484444,
  'name': 'Подслушано Краснодар | Типичный | Радар',
  'category': 'Городское сообщество'},
 {'id': 65140014,
  'name': 'Подслушано Родники',
  'category': 'Городское сообщество'},
 {'id': 161280734,
  'name': 'Подслушано в Назарово',
  'category': 'Городское сообщество'},
 {'id': 157826588,
  'name': 'Подслушано Москва',
  'category': 'Городское сообщество'},
 {'id': 223898245, 'name': 'ПОДСЛУШАНО', 'category': 'Объявления'},
 {'id': 189070626,
  'name': 'Подслушано Беременна в 16',
  'category': 'Философия'}]

In [6]:
filtered_communities_stage_1 = vk_utils.filter_vk_communities_by_category_stop_words(
    communities=new_communities, state_dsn=getenv("POSTGRESQL_DSN"))
print("Отфильтровано сообществ:", len(
    new_communities) - len(filtered_communities_stage_1))
filtered_communities_stage_1[:10]

Отфильтровано сообществ: 10


[{'id': 140549268, 'name': 'Подслушано Каршеринг', 'category': 'Автомобили'},
 {'id': 143277166,
  'name': 'ПОДСЛУШАНО НА СВО',
  'category': 'Дискуссионный клуб'},
 {'id': 224949751,
  'name': 'ПОДСЛУШАНО НА СВО / Выплаты, Поиск пропавших',
  'category': 'Дискуссионный клуб'},
 {'id': 185347562,
  'name': 'Подслушано ФСИН России',
  'category': 'Группа коллег'},
 {'id': 99484444,
  'name': 'Подслушано Краснодар | Типичный | Радар',
  'category': 'Городское сообщество'},
 {'id': 65140014,
  'name': 'Подслушано Родники',
  'category': 'Городское сообщество'},
 {'id': 161280734,
  'name': 'Подслушано в Назарово',
  'category': 'Городское сообщество'},
 {'id': 157826588,
  'name': 'Подслушано Москва',
  'category': 'Городское сообщество'},
 {'id': 223898245, 'name': 'ПОДСЛУШАНО', 'category': 'Объявления'},
 {'id': 147466540,
  'name': 'Подслушано Ярославль',
  'category': 'Городское сообщество'}]

In [7]:
from vk_api import VkApi

with connect_pgsql(getenv("POSTGRESQL_DSN")) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT group_id, reason_discarded "
            "FROM vk_monitor_seen "
            "WHERE reason_discarded = 1 "
            "LIMIT 10"
        )
        discarded = [row[0] for row in cur.fetchall()]

print("Отфильтрованные 1й стадией сообщества:")
api = VkApi(token=getenv("VK_API_KEY"), api_version="5.199").get_api()
discarded_communities = []
if discarded:
    discarded_communities = [group["name"] for group in api.groups.getById(
        group_ids=",".join(map(str, discarded)))["groups"]][:10]
discarded_communities

Отфильтрованные 1й стадией сообщества:


['Подслушано Беременна в 16',
 'Подслушано у бультерьеров',
 'Подслушано про Аквариум',
 'Подслушано Border Collie',
 'Подслушано Битва Экстрасенсов 16+',
 'Подслушано в Снежном',
 'Раменский район. Поиск потерянных животных!',
 'Крафт Место  - шоурум авторских работ',
 'Пугачевский район. Интересное',
 'Автозаводцы - Автозаводский район']

In [8]:
filtered_communities_stage_2 = vk_utils.filter_vk_communities_semantically(api_key=getenv(
    "VK_API_KEY"), communities=filtered_communities_stage_1, state_dsn=getenv("POSTGRESQL_DSN"))
print("Отфильтровано сообществ:", len(
    filtered_communities_stage_1) - len(filtered_communities_stage_2))
filtered_communities_stage_2[:10]

Отфильтровано сообществ: 9


[{'id': 143277166,
  'name': 'ПОДСЛУШАНО НА СВО',
  'category': 'Дискуссионный клуб'},
 {'id': 185347562,
  'name': 'Подслушано ФСИН России',
  'category': 'Группа коллег'},
 {'id': 99484444,
  'name': 'Подслушано Краснодар | Типичный | Радар',
  'category': 'Городское сообщество'},
 {'id': 65140014,
  'name': 'Подслушано Родники',
  'category': 'Городское сообщество'},
 {'id': 161280734,
  'name': 'Подслушано в Назарово',
  'category': 'Городское сообщество'},
 {'id': 157826588,
  'name': 'Подслушано Москва',
  'category': 'Городское сообщество'},
 {'id': 147466540,
  'name': 'Подслушано Ярославль',
  'category': 'Городское сообщество'},
 {'id': 158354930,
  'name': 'Подслушано Самара',
  'category': 'Городское сообщество'},
 {'id': 132750707,
  'name': 'Пермь | Новости | Подслушано',
  'category': 'Городское сообщество'},
 {'id': 39374167,
  'name': 'Подслушано Левобережный Ховрино',
  'category': 'Городское сообщество'}]

In [9]:
with connect_pgsql(getenv("POSTGRESQL_DSN")) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT group_id, reason_discarded "
            "FROM vk_monitor_seen "
            "WHERE reason_discarded = 2 "
            "LIMIT 10"
        )
        discarded = [row[0] for row in cur.fetchall()]

print("Отфильтрованные 2й стадией сообщества:")
api = VkApi(token=getenv("VK_API_KEY"), api_version="5.199").get_api()
discarded_communities = []
if discarded:
    discarded_communities = [group["name"] for group in api.groups.getById(
        group_ids=",".join(map(str, discarded)))["groups"]][:10]
discarded_communities

Отфильтрованные 2й стадией сообщества:


['Подслушано Каршеринг',
 'ПОДСЛУШАНО НА СВО / Выплаты, Поиск пропавших',
 'ПОДСЛУШАНО',
 'Калуга Подслушано',
 'Подслушано Онега',
 'Подслушано ушами',
 'Подслушано Даровской™ Открытая стена',
 'Подслушано Дюртюли',
 'Регион 29 | Информационное агентство']

In [10]:
owner_id = -int("237677627")

api = VkApi(token=getenv("VK_API_KEY"), api_version="5.199").get_api()
posts = api.wall.get(owner_id=owner_id, count=10)["items"]
posts_data = [{"text": post.get("text", "")} for post in posts]

posts_data

[{'text': 'Тест 2'},
 {'text': 'Тест 1'},
 {'text': 'Говорят, что изменения происходят тогда, когда становится слишком неудобно оставаться прежним.\n\nПоследние несколько месяцев стали для меня периодом большой переоценки. Я понял(а), что многие вещи, которые я считал(а) обязательными (стандарты успеха, мнение окружающих, привычка всем угождать), на самом деле просто не мои. Они навязанные, тяжелые и забирают слишком много энергии.\n\nСейчас я учусь говорить «нет» без чувства вины и «да» — своим истинным желаниям, даже если они кажутся кому-то странными или нелогичными. Это путь, и он не самый быстрый, но он мой. 👣\n\nА вы проходили через периоды «пересборки» себя? Как справлялись?\n\n#мысливслух #саморазвитие #путьксебе #перемены'},
 {'text': 'Пора заканчивать это безобразие! ✊\n\nМы уже сто раз говорили про мусор у железнодорожных путей, но воз и ныне там. Мусор лежит месяцами, превращая прилегающую территорию в помойку. Хватит жаловаться в пустоту, пора действовать!\n\nПредлагаю сос

In [12]:
import post_filtering_utils
importlib.reload(post_filtering_utils)

<module 'post_filtering_utils' from '/home/Donut/Nextcloud/Education/02-project/post_filtering_utils.py'>

In [ ]:
filtered_posts_stage_1 = post_filtering_utils.filter_posts_with_faiss(
    posts=posts_data)

print("Отфильтровано постов:", len(posts_data) - len(filtered_posts_stage_1))
filtered_posts_stage_1[:10]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Отфильтровано постов: 6


[{'text': 'Пора заканчивать это безобразие! ✊\n\nМы уже сто раз говорили про мусор у железнодорожных путей, но воз и ныне там. Мусор лежит месяцами, превращая прилегающую территорию в помойку. Хватит жаловаться в пустоту, пора действовать!\n\nПредлагаю составить коллективное обращение в [название ведомства/РЖД/Администрацию]. Если будем писать по одному — нас не услышат. Пишем массово! Кто со мной? Пишите в комментариях 👇\n\n#активисты #чистыйрайон #сделаемгородчище #ждпути #вместемысила'},
 {'text': 'Наконец-то это свершилось! Наводим порядок! 🙌\n\nХотим выразить огромную благодарность коммунальным службам за оперативную работу! Наконец-то был полностью ликвидирован весь этот мусор у железнодорожных путей, который так долго портил нам вид.\n\nДолгое время этот мусор у железнодорожных путей был огромной проблемой для всех жителей района, но теперь на месте бывшей свалки — чистота и порядок. Мы очень рады, что этот мусор у железнодорожных путей больше не беспокоит нас и не портит эколог

In [12]:
filtered_posts_stage_2 = post_filtering_utils.filter_posts_with_llm(
    posts=filtered_posts_stage_1,
    openai_api=getenv("OPENAI_API"),
    openai_api_key=getenv("OPENAI_API_KEY"),
    openai_model=getenv("OPENAI_API_MODEL")
)

print("Отфильтровано постов:", len(
    filtered_posts_stage_1) - len(filtered_posts_stage_2))
filtered_posts_stage_2[:10]

Отфильтровано постов: 2


[{'text': 'Соседи, вы заметили, как сильно изменилась ситуация с мусором у путей? ⚠\n\nЭто уже не просто вопрос красоты. Эти залежи мусора у железной дороги — это рассадник инфекции и грызунов. С ветром этот запах разносится по всем ближайшим домам. К тому же, если это разлетится или кто-то решит развести там костер, последствия будут плачевными.\n\nМы не можем просто закрывать на это глаза. Нужно коллективное обращение в администрацию и к перевозчикам. Кто готов подписаться?\n\n#экология #безопасность #здоровье #соседи #чистыепути'},
 {'text': '«Новый ландшафтный дизайн» нашего района! ✨\n\nЕсли вы решили, что вам не хватает эстетики в жизни, просто посмотрите в сторону ж/д путей. Там у нас развернулась целая экспозиция: пакеты, бутылки и «эксклюзивные» находки, которые лежат там уже не первый месяц. Видимо, это секретный арт-объект, который никто не решается убрать. 🙄\n\nА если серьезно — это просто дно. Сколько можно игнорировать эту свалку прямо под носом у людей?\n\n#эстетика #наш

In [9]:
vk_utils.refresh_vk_monitor_polling_list(
    api_key=getenv("VK_API_KEY"),
    state_dsn=getenv("POSTGRESQL_DSN"),
    openai_api=getenv("OPENAI_API"),
    openai_api_key=getenv("OPENAI_API_KEY"),
    openai_model=getenv("OPENAI_MODEL")
)

True

In [16]:
new_messages = vk_utils.poll_vk_new_messages(
    api_key=getenv("VK_API_KEY"),
    state_dsn=getenv("POSTGRESQL_DSN")
)
new_messages

[{'group_id': 237677627,
  'post_id': 23,
  'text': 'Мусор у железной дороги',
  'latitude': '56.271682008938',
  'longitude': '34.310556247134'},
 {'group_id': 237677627,
  'post_id': 22,
  'text': 'Мусор у железной дороги рядом с Белорусским вокзалом',
  'latitude': None,
  'longitude': None},
 {'group_id': 44250100,
  'post_id': 64973,
  'text': 'Девочки, посдскажите как можно снять боль в спине? Муж походу перестарался с тяжестями в тренажерке. Утром еле встал, ходит полускрюченный. Предлагала ему обезболивающие таблетки, не хочет. Ждет. когда само пройдет, типа не страшно. А мне прям жалко на него смотреть.',
  'latitude': None,
  'longitude': None},
 {'group_id': 46470818,
  'post_id': 898882,
  'text': 'Бригада строителей выполнит: строительство дома, гаража, внутренние работы в доме и квартире. Приемлемые цены.   Тел 8-927-65-35-470',
  'latitude': None,
  'longitude': None}]

In [17]:
filtered_new_messages = post_filtering_utils.filter_posts_with_llm(
    posts=post_filtering_utils.filter_posts_with_faiss(new_messages),
    openai_api=getenv("OPENAI_API"),
    openai_api_key=getenv("OPENAI_API_KEY"),
    openai_model=getenv("OPENAI_MODEL")
)
filtered_new_messages

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'group_id': 237677627,
  'post_id': 23,
  'text': 'Мусор у железной дороги',
  'latitude': '56.271682008938',
  'longitude': '34.310556247134'},
 {'group_id': 237677627,
  'post_id': 22,
  'text': 'Мусор у железной дороги рядом с Белорусским вокзалом',
  'latitude': None,
  'longitude': None}]

In [20]:
import geolocation_utils
importlib.reload(geolocation_utils)

<module 'geolocation_utils' from '/home/Donut/Nextcloud/Education/02-project/geolocation_utils.py'>

In [21]:
filtered_new_messages_with_geo = geolocation_utils.enrich_posts_with_coords(
    posts=filtered_new_messages,
    openai_api=getenv("OPENAI_API"),
    openai_api_key=getenv("OPENAI_API_KEY"),
    openai_model=getenv("OPENAI_MODEL")
)
filtered_new_messages_with_geo

[{'group_id': 237677627,
  'post_id': 23,
  'text': 'Мусор у железной дороги',
  'latitude': '56.271682008938',
  'longitude': '34.310556247134'},
 {'group_id': 237677627,
  'post_id': 22,
  'text': 'Мусор у железной дороги рядом с Белорусским вокзалом',
  'latitude': 55.7763713,
  'longitude': 37.5817009}]